# v0.9 Dataset Preparation
## kishanvavdara + huikang → Format 4 (`<think>…</think>\\boxed{}`)

**Goal**: combine two datasets into a single correct training format covering all 14
competition test categories.

| Dataset | Categories | Rows | Format issue |
|---------|-----------|------|-------------|
| kishanvavdara | 7 (train.csv only) | 4,423 correct | Missing `</think>` + `\\boxed{}` |
| huikang | **14** (all test cats) | 15,159 | Missing `\\boxed{}` for 5 categories |

**Format 4** (what we build toward):
```
<|im_start|>assistant
<think>          ← auto-prepended by chat template
{reasoning trace}
</think>
\\boxed{answer}
```
In training JSONL the assistant content is `{trace}\n</think>\n\\boxed{answer}`.

In [1]:
import csv, json, random, re
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import display, HTML

WORKSPACE = Path("..")
random.seed(42)

C = {
    "green":  "#c8f7c5",
    "red":    "#ffd6d6",
    "yellow": "#fff3cd",
    "blue":   "#d0e8ff",
    "gray":   "#f0f0f0",
    "orange": "#ffe4b5",
    "teal":   "#d0f4f4",
    "purple": "#ead9ff",
}

def hl(text, colour, bold=False):
    w = 'font-weight:bold;' if bold else ''
    return (f'<span style="background:{colour};{w}'
            f'padding:1px 4px;border-radius:3px;font-family:monospace">{text}</span>')

def pre(content, title="", border_colour="#ccc"):
    t = f'<div style="font-size:0.8em;font-weight:bold;color:#555;margin-bottom:3px">{title}</div>' if title else ''
    return HTML(
        f'{t}<div style="background:#fafafa;border:1px solid {border_colour};'
        f'border-radius:6px;padding:14px 18px;font-family:monospace;'
        f'font-size:0.85em;line-height:1.7;white-space:pre-wrap">'
        f'{content}</div>'
    )

def section(title, subtitle=""):
    sub = f'<div style="color:#555;margin-top:2px;font-size:0.9em">{subtitle}</div>' if subtitle else ''
    return HTML(f'<h3 style="margin-bottom:4px;border-left:4px solid #444;padding-left:10px">{title}</h3>{sub}')

def html_escape(s):
    return s.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")

def highlight_format4(text, max_chars=700):
    """Colour-code <think>, </think>, \\boxed{} in a formatted assistant string."""
    preview = text[:max_chars] + (" …" if len(text) > max_chars else "")
    safe = html_escape(preview)
    safe = safe.replace("&lt;think&gt;",  hl("&lt;think&gt;",  C["green"], bold=True))
    safe = safe.replace("&lt;/think&gt;", hl("&lt;/think&gt;", C["green"], bold=True))
    safe = re.sub(r'(\\boxed\{[^}]*\})',
                  lambda m: hl(m.group(1), C["blue"], bold=True), safe)
    return safe

print("Helpers ready.")

Helpers ready.


---
## Section 0 — How Do We Know There Are 14 Categories?

`train.csv` exposes only **6** problem types. The competition test set evaluates **14**.
The huikang corpus reveals this because it covers problems from both the training set
and the competition test set (which includes 8 additional problem types never seen in training).

In [2]:
with open(WORKSPACE / "data" / "train.csv") as f:
    train_rows = list(csv.DictReader(f))
with open(WORKSPACE / "data" / "v0.4_train.jsonl") as f:
    huikang_rows = [json.loads(l) for l in f]

huikang_cats = Counter(r.get('category','?') for r in huikang_rows)
train_ids = {r['id'] for r in train_rows}

in_train = Counter()
test_only = Counter()
for r in huikang_rows:
    cat = r.get('category','?')
    if r.get('id','') in train_ids:
        in_train[cat] += 1
    else:
        test_only[cat] += 1

all_cats = sorted(huikang_cats.keys())

hdr = "".join(f"<th style='padding:6px 12px;background:#f0f0f0;text-align:left'>{h}</th>"
              for h in ["Category", "Total in huikang", "In train.csv", "Test-only", "In train.csv?"])
rows_html = ""
for cat in all_cats:
    n_total = huikang_cats[cat]
    n_train = in_train[cat]
    n_test  = test_only[cat]
    in_tr   = "✅" if n_train > 0 else "❌ test only"
    bg = "" if n_train > 0 else f"background:{C['orange']};"
    rows_html += (
        f"<tr style='{bg}'>"
        f"<td style='padding:6px 12px;font-family:monospace'>{cat}</td>"
        f"<td style='padding:6px 12px;text-align:right'>{n_total:,}</td>"
        f"<td style='padding:6px 12px;text-align:right'>{n_train:,}</td>"
        f"<td style='padding:6px 12px;text-align:right'>{n_test:,}</td>"
        f"<td style='padding:6px 12px;text-align:center'>{in_tr}</td>"
        f"</tr>"
    )
display(HTML(
    f"<table style='border-collapse:collapse;font-size:0.9em'>"
    f"<thead><tr>{hdr}</tr></thead><tbody>{rows_html}</tbody></table>"
))
print(f"\n{len(all_cats)} categories in huikang | train.csv has {len(train_ids):,} problems")
print(f"{sum(test_only.values()):,} huikang rows are test-set problems (categories never seen in train.csv)")

Category,Total in huikang,In train.csv,Test-only,In train.csv?
bit_manipulation,"1,932","1,281",651,✅
cipher,"1,659","1,487",172,✅
concatenation,"1,422",0,"1,422",❌ test only
cryptarithm_deduce,123,53,70,✅
cryptarithm_guess,175,156,19,✅
equation_numeric_deduce,603,511,92,✅
equation_numeric_guess,168,132,36,✅
gravity,920,892,28,✅
lstrip,286,0,286,❌ test only
matching,"4,316",0,"4,316",❌ test only



14 categories in huikang | train.csv has 9,500 problems
9,150 huikang rows are test-set problems (categories never seen in train.csv)


In [3]:
# One prompt snippet per category — what does each problem type look like?
seen = {}
for r in huikang_rows:
    cat = r.get('category', '?')
    if cat not in seen:
        seen[cat] = r['prompt']

for cat in all_cats:
    prompt = seen.get(cat, '')
    prompt_short = html_escape(prompt[:350]) + (" …" if len(prompt) > 350 else "")
    in_tr_tag = hl("train.csv", C["green"], bold=True) if in_train[cat] > 0 else hl("test-only", C["orange"], bold=True)
    content = (
        f"{hl('CATEGORY', C['gray'], bold=True)} {hl(cat, C['teal'], bold=True)}  "
        f"{in_tr_tag}  ({huikang_cats[cat]:,} huikang rows)\n\n"
        f"{prompt_short}"
    )
    display(pre(content, border_colour=C["teal"] if in_train[cat] > 0 else C["orange"]))
    print()

---
## Section 1 — Conversion Functions

**Huikang**: `response` already ends with `</think>` (sometimes also `\\boxed{}`).  
The chat template adds `<think>\n` automatically, so `response` is the full assistant content.  
For 5 categories that end with just `</think>`, we extract the last output line as `\\boxed{}`.

**kishanvavdara**: `generated` = reasoning trace.  
We append `\n</think>\n\\boxed{correct_answer}`.

In [4]:
PROMPT_SUFFIX = "\nPlease put your final answer inside \\boxed{}."

TYPE_TO_BUCKET = {
    "bit_manipulation":        "bit_like",
    "cipher":                  "cipher_like",
    "concatenation":           "other",
    "cryptarithm_deduce":      "equation_like",
    "cryptarithm_guess":       "equation_like",
    "equation_numeric_deduce": "equation_like",
    "equation_numeric_guess":  "equation_like",
    "equation_numeric":        "equation_like",
    "equation_symbolic":       "equation_like",
    "gravity":                 "other",
    "lstrip":                  "other",
    "matching":                "other",
    "numeral":                 "numeral_like",
    "spelling":                "cipher_like",
    "splitting":               "other",
    "unit_conversion":         "unit_like",
}

THINK_ONLY_CATS = {"concatenation", "lstrip", "matching", "spelling", "splitting"}


def extract_think_only_answer(response: str, category: str) -> str:
    resp = response.rstrip()
    if resp.endswith('</think>'):
        resp = resp[:-len('</think>')].rstrip()
    lines = [l.strip() for l in resp.split('\n') if l.strip()]
    if not lines:
        return ""
    if category == "matching":
        bests = re.findall(r'Best:\s*(.+?):\s*\d+', resp)
        return bests[-1].strip() if bests else lines[-1]
    last = lines[-1]
    if ' -> ' in last:
        return last.split(' -> ', 1)[1].strip()
    return last


def build_huikang_example(row: dict) -> dict:
    cat  = row.get('category', '?')
    resp = row['response'].strip()
    think_pos = resp.rfind('</think>')
    has_boxed_after = bool(think_pos >= 0 and re.search(r'\\boxed\{', resp[think_pos:]))
    if not has_boxed_after and cat in THINK_ONLY_CATS:
        answer = extract_think_only_answer(resp, cat)
        if resp.endswith('</think>'):
            resp = resp + f'\n\\boxed{{{answer}}}'
    return {
        "messages": [
            {"role": "user",      "content": row['prompt'] + PROMPT_SUFFIX},
            {"role": "assistant", "content": resp},
        ],
        "bucket":   TYPE_TO_BUCKET.get(cat, "other"),
        "category": cat,
        "source":   "huikang",
        "id":       row.get('id', ''),
    }


def build_kishanvavdara_example(row: dict) -> dict:
    cat       = row.get('problem type', '?')
    generated = row['generated'].strip()
    answer    = row['correct answer'].strip()
    asst = f"{generated}\n</think>\n\\boxed{{{answer}}}"
    return {
        "messages": [
            {"role": "user",      "content": row['prompt'].strip() + PROMPT_SUFFIX},
            {"role": "assistant", "content": asst},
        ],
        "bucket":   TYPE_TO_BUCKET.get(cat, "other"),
        "category": cat,
        "source":   "kishanvavdara",
        "id":       row.get('id', ''),
    }


huikang_examples = [build_huikang_example(r) for r in huikang_rows]
print(f"Huikang examples built: {len(huikang_examples):,}")

Huikang examples built: 15,159


In [5]:
try:
    import kagglehub
    print("Downloading kishanvavdara/nemotron-reasoning-traj …")
    path = kagglehub.dataset_download("kishanvavdara/nemotron-reasoning-traj")
    csv_path = next(Path(path).glob("**/*.csv"), None)
except Exception as e:
    print(f"kagglehub failed ({e}), trying local cache …")
    csv_path = next(Path("/home/msusol/.cache/kagglehub/datasets/kishanvavdara")
                    .glob("**/nemotron_traj.csv"), None)

with open(csv_path, encoding="utf-8") as f:
    kv_rows = [r for r in csv.DictReader(f) if r.get("correctness","").lower() == "true"]

kv_examples = [build_kishanvavdara_example(r) for r in kv_rows]
print(f"kishanvavdara correct rows : {len(kv_rows):,}")
print(f"kishanvavdara examples built: {len(kv_examples):,}")

kishanvavdara correct rows : 4,423
kishanvavdara examples built: 4,423


---
## Section 1.5 — Before → After: Row Conversion Examples

Three cases: huikang (already has `\\boxed{}`), huikang think-only (answer extracted from last line), and kishanvavdara.

In [6]:
def show_before_after(title, before_label, before_fields, after_ex, border="#888"):
    """Render a side-by-side before/after conversion panel."""
    asst = after_ex["messages"][1]["content"]
    full_out = "<think>\n" + asst

    # BEFORE panel
    before_html = ""
    for k, v in before_fields.items():
        v_safe = html_escape(str(v)[:400]) + (" …" if len(str(v)) > 400 else "")
        before_html += f"{hl(k, C['gray'], bold=True)}\n{v_safe}\n\n"

    # AFTER panel
    after_html = highlight_format4(full_out, max_chars=600)

    display(HTML(
        f"<h4 style='margin-bottom:6px'>{title}</h4>"
        f"<div style='display:grid;grid-template-columns:1fr 1fr;gap:12px'>"

        f"<div>"
        f"<div style='font-size:0.8em;font-weight:bold;color:#555;margin-bottom:4px'>"
        f"BEFORE — {before_label}</div>"
        f"<div style='background:#fff8f0;border:1px solid {C['orange']};border-radius:6px;"
        f"padding:12px;font-family:monospace;font-size:0.82em;line-height:1.6;"
        f"white-space:pre-wrap;min-height:120px'>{before_html.rstrip()}</div>"
        f"</div>"

        f"<div>"
        f"<div style='font-size:0.8em;font-weight:bold;color:#555;margin-bottom:4px'>"
        f"AFTER — assistant content (chat template adds &lt;think&gt;)</div>"
        f"<div style='background:#f0fff0;border:1px solid {C['green']};border-radius:6px;"
        f"padding:12px;font-family:monospace;font-size:0.82em;line-height:1.6;"
        f"white-space:pre-wrap;min-height:120px'>{after_html}</div>"
        f"</div>"

        f"</div><br>"
    ))


# ── Case 1: huikang with \boxed{} already present ─────────────────────────
hk_boxed_row = next(r for r in huikang_rows
                    if r.get('category') == 'bit_manipulation')
hk_boxed_ex  = build_huikang_example(hk_boxed_row)
show_before_after(
    "Case 1 — huikang (category already has \\boxed{} after &lt;/think&gt;)",
    "huikang raw fields",
    {
        "category": hk_boxed_row.get("category"),
        "response (last 300 chars)": hk_boxed_row["response"][-300:],
    },
    hk_boxed_ex,
)

# ── Case 2: huikang think-only (\boxed{} extracted from last line) ────────
hk_to_row = next(r for r in huikang_rows
                 if r.get('category') == 'splitting')
hk_to_ex  = build_huikang_example(hk_to_row)
last_line = [l.strip() for l in hk_to_row['response'].split('\n') if l.strip()][-2]  # -2 = before </think>
show_before_after(
    "Case 2 — huikang think-only (\\boxed{} extracted from last response line)",
    "huikang raw fields",
    {
        "category": hk_to_row.get("category"),
        "response (last 3 lines)": "\n".join(
            [l for l in hk_to_row['response'].split('\n') if l.strip()][-3:]
        ),
        "→ extracted answer": extract_think_only_answer(hk_to_row['response'], 'splitting'),
    },
    hk_to_ex,
)

# ── Case 3: kishanvavdara ─────────────────────────────────────────────────
kv_raw_row = kv_rows[0]
kv_ex      = build_kishanvavdara_example(kv_raw_row)
show_before_after(
    "Case 3 — kishanvavdara (wrap generated trace with &lt;/think&gt; + \\boxed{})",
    "kishanvavdara raw fields",
    {
        "problem type": kv_raw_row.get("problem type"),
        "correct answer": kv_raw_row.get("correct answer"),
        "generated (last 200 chars)": kv_raw_row["generated"][-200:],
    },
    kv_ex,
)

---
## Section 2 — Format Validation Stats

Check every converted example for the required structural elements.

In [7]:
all_examples = huikang_examples + kv_examples

def check_format(ex):
    asst = ex["messages"][1]["content"]
    think_pos = asst.rfind('</think>')
    boxed_pos = asst.rfind('\\boxed{')
    return {
        "has_think_close":    think_pos >= 0,
        "has_boxed":          boxed_pos >= 0,
        "boxed_after_think":  think_pos >= 0 and boxed_pos > think_pos,
        "approx_tokens":      len(asst) // 4,
        "source":             ex["source"],
        "category":           ex["category"],
    }

checks = [check_format(ex) for ex in all_examples]
n = len(checks)

n_think = sum(c["has_think_close"]   for c in checks)
n_boxed = sum(c["has_boxed"]         for c in checks)
n_both  = sum(c["boxed_after_think"] for c in checks)

def pct(k): return f"{100*k/n:.1f}%"

summary = [
    ("Total examples",                         n,       "—"),
    ("Has &lt;/think&gt;",                      n_think, pct(n_think)),
    ("Has \\boxed{}",                           n_boxed, pct(n_boxed)),
    ("\\boxed{} after &lt;/think&gt; ✅ Format 4", n_both,  pct(n_both)),
    ("Incomplete ❌",                            n-n_both, pct(n-n_both)),
]
hdr_h = "".join(f"<th style='padding:6px 14px;background:#f0f0f0'>{h}</th>"
                for h in ["Check", "Count", "% of total"])
body_h = "".join(
    f"<tr><td style='padding:6px 14px;font-family:monospace'>{l}</td>"
    f"<td style='padding:6px 14px;text-align:right'>{c:,}</td>"
    f"<td style='padding:6px 14px;text-align:right'>{p}</td></tr>"
    for l, c, p in summary
)
display(HTML(f"<table style='border-collapse:collapse;font-size:0.9em'>"
             f"<thead><tr>{hdr_h}</tr></thead><tbody>{body_h}</tbody></table>"))

lens = sorted(c["approx_tokens"] for c in checks)
print(f"\nApprox assistant token length:")
print(f"  p25={lens[n//4]}  p50={lens[n//2]}  p90={lens[9*n//10]}  p95={lens[19*n//20]}  p99={lens[99*n//100]}  max={lens[-1]}")
print(f"  → max_seq_length=8192 covers p99 ({lens[99*n//100]} tokens + prompt overhead)")

Check,Count,% of total
Total examples,"19,582",—
Has </think>,"19,582",100.0%
Has \boxed{},"19,582",100.0%
\boxed{} after </think> ✅ Format 4,"19,582",100.0%
Incomplete ❌,0,0.0%



Approx assistant token length:
  p25=167  p50=779  p90=2562  p95=3090  p99=5383  max=7579
  → max_seq_length=8192 covers p99 (5383 tokens + prompt overhead)


In [8]:
# Per-category breakdown
cat_stats = defaultdict(lambda: {"total":0, "boxed_after":0, "tokens":[]})
for c in checks:
    s = cat_stats[c["category"]]
    s["total"]       += 1
    s["boxed_after"]  += int(c["boxed_after_think"])
    s["tokens"].append(c["approx_tokens"])

hdr = "".join(f"<th style='padding:6px 12px;background:#f0f0f0;text-align:left'>{h}</th>"
              for h in ["Category", "Total", "Format 4 ✅", "% correct", "p50 tokens", "In train.csv?"])
body = ""
for cat in sorted(cat_stats.keys()):
    s    = cat_stats[cat]
    tok  = sorted(s["tokens"])
    p50  = tok[len(tok)//2]
    pct_ok = 100 * s["boxed_after"] / s["total"]
    in_tr  = "✅" if in_train[cat] > 0 else "❌"
    bg = "" if pct_ok == 100 else f"background:{C['yellow']};"
    body += (
        f"<tr style='{bg}'>"
        f"<td style='padding:5px 12px;font-family:monospace'>{cat}</td>"
        f"<td style='padding:5px 12px;text-align:right'>{s['total']:,}</td>"
        f"<td style='padding:5px 12px;text-align:right'>{s['boxed_after']:,}</td>"
        f"<td style='padding:5px 12px;text-align:right'>{pct_ok:.0f}%</td>"
        f"<td style='padding:5px 12px;text-align:right'>{p50:,}</td>"
        f"<td style='padding:5px 12px;text-align:center'>{in_tr}</td>"
        f"</tr>"
    )
display(HTML(f"<table style='border-collapse:collapse;font-size:0.9em'>"
             f"<thead><tr>{hdr}</tr></thead><tbody>{body}</tbody></table>"))

Category,Total,Format 4 ✅,% correct,p50 tokens,In train.csv?
bit_manipulation,"2,060","2,060",100%,"2,189",✅
cipher,"2,142","2,142",100%,"1,557",✅
concatenation,"1,422","1,422",100%,766,❌
cryptarithm_deduce,123,123,100%,318,✅
cryptarithm_guess,175,175,100%,330,✅
equation_numeric,176,176,100%,"3,132",❌
equation_numeric_deduce,603,603,100%,"2,620",✅
equation_numeric_guess,168,168,100%,"2,741",✅
equation_symbolic,2,2,100%,"5,781",❌
gravity,"1,855","1,855",100%,"1,090",✅


---
## Section 3 — Random Samples with Highlighting

One sample per category, colour-coded.  
**Green** = `<think>`/`</think>` · **Blue** = `\\boxed{answer}` · **Orange** = huikang · **Teal** = kishanvavdara

In [9]:
by_cat = defaultdict(list)
for ex in all_examples:
    by_cat[ex["category"]].append(ex)

for cat in sorted(by_cat.keys()):
    ex     = random.choice(by_cat[cat])
    user   = ex["messages"][0]["content"]
    asst   = ex["messages"][1]["content"]
    source = ex["source"]
    n_tok  = len(asst) // 4

    user_safe = html_escape(user[:300]) + (" …" if len(user) > 300 else "")
    src_tag   = hl(source, C["orange"] if source=="huikang" else C["teal"], bold=True)
    in_tr_tag = hl("train.csv", C["green"]) if in_train[cat] > 0 else hl("test-only", C["orange"])

    full_out = "<think>\n" + asst
    asst_hl  = highlight_format4(full_out, max_chars=600)

    content = (
        f"{hl('USER', C['gray'], bold=True)}\n"
        f"{user_safe}\n\n"
        f"{hl(f'ASSISTANT (~{n_tok} tokens)', C['gray'], bold=True)} {src_tag}\n"
        f"{asst_hl}"
    )
    display(section(f"{cat}", f"id={ex['id']}  bucket={ex['bucket']}  {in_tr_tag}"))
    display(pre(content, border_colour=C["green"]))
    print()

---
## Section 4 — Save v0.9 Train / Valid Split

huikang (15,159) + kishanvavdara (4,423) = **19,582 examples** across **14 categories**.

In [10]:
VALID_FRAC = 0.05

shuffled = all_examples.copy()
random.shuffle(shuffled)

n_valid   = max(1, int(len(shuffled) * VALID_FRAC))
valid_set = shuffled[:n_valid]
train_set = shuffled[n_valid:]

out_train = WORKSPACE / "data" / "v0.9_train.jsonl"
out_valid = WORKSPACE / "data" / "v0.9_valid.jsonl"

with open(out_train, "w", encoding="utf-8") as f:
    for ex in train_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")
with open(out_valid, "w", encoding="utf-8") as f:
    for ex in valid_set:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

train_src = Counter(ex["source"]   for ex in train_set)
train_cat = Counter(ex["category"] for ex in train_set)

print(f"Total  : {len(all_examples):,}")
print(f"Train  : {len(train_set):,}")
print(f"Valid  : {len(valid_set):,}")
print(f"Sources: {dict(train_src)}")
print(f"Cats   : {dict(sorted(train_cat.items()))}")
print(f"\nSaved → {out_train}")
print(f"Saved → {out_valid}")

display(HTML(f"""
<div style='font-family:sans-serif;font-size:0.9em;line-height:1.8;
            background:{C['gray']};padding:16px 20px;border-radius:8px;margin-top:12px'>
<b>v0.9 data ready.</b><br>
• <b>{len(train_set):,} train</b> + <b>{len(valid_set):,} valid</b> across <b>14 categories</b><br>
• huikang: {train_src['huikang']:,} — all 14 test categories (algorithmic CoT)<br>
• kishanvavdara: {train_src['kishanvavdara']:,} — 7 train categories (Nemotron-30B CoT, correctness-filtered)<br>
• Format: <code>{{trace}}\n&lt;/think&gt;\n\\boxed{{answer}}</code> (chat template prepends &lt;think&gt;)<br>
• <b>Next</b>: SFT warmstart from v0.5-sft-unsloth, max_seq_length=8192, ~500 steps
</div>
"""))

Total  : 19,582
Train  : 18,603
Valid  : 979
Sources: {'huikang': 14385, 'kishanvavdara': 4218}
Cats   : {'bit_manipulation': 1942, 'cipher': 2042, 'concatenation': 1358, 'cryptarithm_deduce': 117, 'cryptarithm_guess': 165, 'equation_numeric': 166, 'equation_numeric_deduce': 575, 'equation_numeric_guess': 158, 'equation_symbolic': 2, 'gravity': 1753, 'lstrip': 274, 'matching': 4108, 'numeral': 2008, 'spelling': 570, 'splitting': 1347, 'unit_conversion': 2018}

Saved → ../data/v0.9_train.jsonl
Saved → ../data/v0.9_valid.jsonl
